# Clustering - K-Means
Aprendizaje no supervisado sobre defunciones de Buenos Aires

## Setup y carga de datos limpios

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etl.main import run as run_etl

df = run_etl('../data/raw/defunciones-ocurridas-y-registradas-en-la-republica-argentina-entre-los-anos-2005-2022.csv')
df.shape

## No Supervisado

In [ ]:
import sklearn.cluster as cluster
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

## Método del Codo

In [ ]:
df_piModNoSup = df.copy()

ordenEdad = ['De a 0 a 14 anios','De 15 a 34 anios',
            'De 35 a 54 anios','De 55 a 74 anios',
            'De 75 anios y mas']
ordEdad = OrdinalEncoder(categories=[ordenEdad])
df_piModNoSup['grupo_edad'] = ordEdad.fit_transform(df_piModNoSup[['grupo_edad']])


X_real = df_piModNoSup[['grupo_edad',	'cantidad', 'anio']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_real)

inercia = []

for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(X_scaled)
    inercia.append(model.inertia_)
# Graficamos los resultados
plt.plot(range(1, 11), inercia, marker='o')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo')
plt.show()

## Entrenamiento K-Means con K=4

In [ ]:
kmeans_final = KMeans(n_clusters=4, n_init='auto', random_state=42)
# n_init='auto' , co auto se asegura que la "Posicion" de los puntos centrales de cada grupo, sean optimos

clusters_real = kmeans_final.fit_predict(X_scaled) # Prediccion

## Estadísticas por cluster

In [ ]:
df_analisis = X_real.copy()
df_analisis['Color_Cluster'] = clusters_real

# Sacamos el promedio real de cada grupo
# Se utiliza para interpretar los grupos
print(df_analisis.groupby('Color_Cluster').mean())

## Calidad de la segmentación

In [ ]:
score = silhouette_score(X_scaled, clusters_real)
print(f"Calidad de la segmentación (Silhouette Score): {score:.4f}")